Finale CQs

Imports

In [40]:
import os
import json
import random
import time
from mistralai import Mistral

Define

In [ ]:
# Input + Output paths
input_path = r"C:\Users\cdoering\Downloads\filtered_output.json"
output_dir = "competency_questions_output"
os.makedirs(output_dir, exist_ok=True)

# Gemeinsame Ausgabedatei
combined_output_path = os.path.join(output_dir, "competency_questions_all_documents_neu.txt")

# JSON laden
with open(input_path, "r", encoding="utf-8") as f:
    documents = json.load(f)

api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
model = "open-mistral-nemo"
client = Mistral(api_key=api_key)
SEED = 69
SAMPLE_SIZE = 200
TEMPERATURE = 0.69

Prompt generator

In [42]:
ontology_purpose = "To build an ontology for legal and regulatory compliance analysis from legal documents, more specifically judgments."

system_message = """You are an experienced ontology and knowledge engineer. Your task is to create competency questions based on documents that will be used in a later stage to create an ontology. Together with the document, you are given a short purpose description for the ontology that must be created.

Remember the definition and characteristics of competency questions:
Competency questions (CQs) are specific questions that an ontology should be able to answer once it is complete. Use them to define the scope and validate the design of the ontology.

Follow these key aspects when generating competency questions:

1. Align questions with the ontology purpose: Reflect the domain and purpose, and address critical use cases.
2. Make questions clear and unambiguous: Use concise, natural language to avoid misunderstandings.
3. Ensure questions are specific and testable: Focus on precise aspects of the domain and make sure they can be validated with data or reasoning.
4. The questions should be formulated in a way that they can be answered with the information provided in the document. Use the following question forms:
   Unter welchen Voraussetzungen..., Unter welchen Bedingungen..., Wer ist verpflichtet..., Welche Rechte hat..., Welche Rolle spielt..., Welche Rechtsgrundlage... etc.
5. The questions should concern the Rechtsgrundlagen, Tatbestandsmerkmale, Voraussetzungen, einschlägige Paragraphen, Rechtsfolgen and Bedingungen!
6. Do NOT ask questions that are too general or too specific. The questions should be relevant to the ontology purpose.
7. Do NOT ask case-specific or overly narrow questions. The following expressions are NOT allowed in the questions: "im vorliegenden Fall...", "im vorliegenden Rechtsstreit",  "im konkreten Rechtsstreit", "in diesem Fall", "in diesem Rechtsstreit",  etc.
The questions must focus on general legal rules, not on the specific facts of a particular case.
8. Avoid names, dates, individual documents, clauses, or parties involved in the case.


Example output of good questions:

Frage: Welche Voraussetzungen müssen erfüllt sein, damit ein nicht im Grundbuch eingetragener Eigentümer eine (Rück-)Auflassung verlangen kann?
Quelle: ["§ 894 BGB bietet dem nicht im Grundbuch eingetragenen Eigentümer auch die Möglichkeit, (Rück-)Auflassung zu verlangen."]

Frage: Welche Rechtsgrundlage gilt für die Zurechnung von Beratungsfehlern eines Dritten auf eine Bank?
Quelle: ["Da die A. AG nicht im Pflichtenkreis der Beklagten tätig geworden sei, scheide eine Zurechnung etwaiger Beratungsfehler der A. AG nach § 278 BGB aus."]

"""

def build_user_prompt(purpose, text):
    return f"""
        Abstract description of the document contents:
        These documents are court rulings (Urteile) from the German Federal Court of Justice (Bundesgerichtshof), issued between 2000 and 2020 and classified as higher court jurisprudence (obere Rechtsprechung). 
        The rulings cover a variety of legal areas including summarized legal themes, civil procedure law (Zivilverfahrensrecht), general contract law (SchuldrechtAT), damages (Schadensersatz), commercial and corporate law (Handelsrecht Gesellschaftsrecht), general provisions of the German Civil Code (BGBAT), family law (Familienrecht), tenancy and lease law (Miete Pacht), property law (Sachenrecht), insurance law (Versicherungsrecht), purchase, exchange, and leasing (Kauf Tausch Leasing), inheritance and gift law (Erbschaft Schenkung), IT law (EDV-Recht), obligations (Schuldverhältnisse), contracts for work and services (Werkvertrag), residential property law (Wohnungseigentum), miscellaneous law (Sonstiges Recht), and travel contract law (Reisevertrag).

        General purpose of the ontology:
        {purpose}

        Document:
        {text[:10000]}

        Please generate 5-10 competency questions in German that reflects a specific legal obligation, condition, actor or rule. Use different question forms like Wer, Was, Wie, Warum, Wann etc. Do not repeat structure.

        Your task:
        •⁠  ⁠Generate 5-10 competency questions (CQs) in German based on this document.
        •⁠  ⁠For each CQ, include the original sentence(s) you used as justification.
        •⁠  ⁠Focus on legal reasoning by formulating questions that involve legal interpretation, applicability of rules, conditions, or exceptions.
        •⁠  ⁠Include obligations by addressing duties, rights, or responsibilities of parties involved.
        •⁠  ⁠Use facts by referencing concrete legal situations, case details, or normative statements from the document.
        •⁠  ⁠Only list questions and citations. Do not explain or comment.
        •⁠  Remember to avoid case-specific or overly narrow questions. The questions should focus on general legal rules, not on the specific facts of a particular case.
        """


Helper functions

In [43]:
def extract_text(doc_data):
    text_data = doc_data.get("text", {}).get("entscheidungsinhalt", {})
    leitsatz = text_data.get("leitsatz", "")
    tenor = text_data.get("tenor", "")
    tatbestand = text_data.get("tatbestand", "")
    gruende_list = text_data.get("gruende", {}).get("gruende", [])

    return "\n\n".join(part for part in [
        "Leitsatz:\n" + leitsatz.strip() if isinstance(leitsatz, str) else "",
        "Entscheidungsformel:\n" + tenor.strip() if isinstance(tenor, str) else "",
        "Sachverhalt:\n" + tatbestand.get("#text", "").strip() if isinstance(tatbestand, dict) else tatbestand.strip() if isinstance(tatbestand, str) else "",
        "Begründung:\n" + "\n".join(gruende_list).strip() if gruende_list else ""
    ] if part).strip()

def format_output(doc_id, output_text, rthema):
    lines = [line.strip() for line in output_text.splitlines() if line.strip()]
    fragen = [line for line in lines if "**Frage:**" in line]
    quellen = [line for line in lines if "**Quelle:**" in line]

    formatted = f"Dokument: {doc_id}\n"
    formatted += f"Rechtsthema: {', '.join(rthema)}\n\n"
    
    if fragen and quellen:
        for frage, quelle in zip(fragen, quellen):
            formatted += f"{frage}\n{quelle}\n\n"
    else:
        formatted += "No questions or sources generated.\n\n"
    return formatted


In [44]:
# Clear Document
open(combined_output_path, "w", encoding="utf-8").close()
# Zufällige Auswahl von SAMPLE_SIZE Dokumenten verarbeiten
random.seed(SEED)
sampled_documents = documents
#sampled_documents = dict(random.sample(list(documents.items()), SAMPLE_SIZE))

# === Processing Documents ===
for doc_id, doc_data in sampled_documents.items():
    text = extract_text(doc_data)

    user_prompt = build_user_prompt(ontology_purpose, text)

    try:
        response = client.chat.complete(
            model=model,
            random_seed=SEED,
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_prompt},
            ],
            temperature=TEMPERATURE,
        )

        output = response.choices[0].message.content.strip()
        # === rthema extrahieren ===
        rthema = doc_data.get("allgemeine-angaben", {}).get("rthema", [])
        # === Formatierte Ausgabe inkl. rthema ===
        formatted_entry = format_output(doc_id, output, rthema)

        with open(combined_output_path, "a", encoding="utf-8") as f_out:
            f_out.write(formatted_entry)

        print(f"Gespeichert: {doc_id}")
        time.sleep(0.01)

    except Exception as e:
        print(f"[ERROR] Fehler bei {doc_id}: {e}")


Gespeichert: 1590043.xml
Gespeichert: 0962207.xml
Gespeichert: 1540712.xml
Gespeichert: 1665264.xml
Gespeichert: 1557508.xml
Gespeichert: 1582321.xml
Gespeichert: 1520780.xml
Gespeichert: 1572978.xml
Gespeichert: 1550273.xml
Gespeichert: 1540060.xml
Gespeichert: 0962213.xml
Gespeichert: 1541418.xml
Gespeichert: 1594231.xml
Gespeichert: 1538865.xml
Gespeichert: 1527945.xml
Gespeichert: 1524294.xml
Gespeichert: 1586635.xml
Gespeichert: 1589506.xml
Gespeichert: 1533918.xml
Gespeichert: 1669462.xml
Gespeichert: 1597076.xml
Gespeichert: 1540048.xml
Gespeichert: 1672056.xml
Gespeichert: 1572788.xml
Gespeichert: 0961040.xml
Gespeichert: 1580278.xml
Gespeichert: 1657598.xml
Gespeichert: 0965540.xml
Gespeichert: 1538859.xml
Gespeichert: 1536647.xml
Gespeichert: 1653715.xml
Gespeichert: 1503824.xml
Gespeichert: 1572763.xml
Gespeichert: 1576505.xml
Gespeichert: 1655402.xml
Gespeichert: 1539212.xml
Gespeichert: 1521449.xml
Gespeichert: 1529015.xml
Gespeichert: 1594594.xml
Gespeichert: 1532353.xml


In [48]:
import json

combined_output_path = r"C:\Users\cdoering\OneDrive - IW\Dokumente\Capstone\competency_questions_all_documents.txt"

with open(combined_output_path, "w", encoding="utf-8") as f:
    json.dump(competency_questions_all_documents.txt, f, indent=2, ensure_ascii=False)


NameError: name 'competency_questions_all_documents' is not defined